# 05 · Train DeBERTa-v3 on SQuAD 2.0

**Experiment 1 of `qa-experiments`.** Same pipeline as the DistilBERT run, with a stronger pretrained model,
`microsoft/deberta-v3-base` (12 layers, 184 M parameters; DistilBERT has 6 and 66 M). The goal is to beat
`deepset/roberta-base-squad2`: **79.9 EM / 82.9 F1** on the dev set. DistilBERT got 64.1 / 67.4.

Run it in Google Colab with **Runtime → Change runtime type → T4 GPU**.

1. **Trial run** (sections 0–1, a few minutes): 50 steps on 1% of the data. Checks that DeBERTa trains on a T4
   without running out of memory or producing `nan` losses, and estimates how long the full run will take.
2. **Full run** (sections 2–4, probably several hours): 2 epochs on all the data, checkpoints to Google Drive,
   the model uploaded to the Hugging Face Hub as `Ingsyling/deberta-v3-squad2`, then scored on the dev set.

**Do the trial run first and send its output before starting the full run.**

All settings are in `configs/deberta.yaml`.

## 0. Setup
Clones this repo and installs the requirements. Safe to run more than once.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Davidfaleye360/qa-experiments.git"
REPO_DIR = "/content/qa-experiments"

if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally: work from the repo root

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import transformers, datasets, torch

print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
if int(transformers.__version__.split(".")[0]) >= 5:
    raise RuntimeError("transformers 5.x is still loaded. Use Runtime -> Restart session, then run the setup cell again.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then run the setup cell again.")
    print("The smoke test still works without one, just more slowly.")

## 1. Trial run

`configs/smoke.yaml` on top of `configs/deberta.yaml`: 50 weight updates on 1% of the training data, with
everything else as in the full run (batch size, fp16, learning rate). What to look for:

- **`nan` in the loss.** DeBERTa-v3 is known to be unstable in fp16 sometimes. Then the result is FAIL.
- **"CUDA out of memory".** Then the batch has to get smaller.
- **RESULT: PASS** at the end, and the loss falling from about 5.9.

You'll see a warning that some weights were "newly initialized": that's the new answer layer, as before.

In [ ]:
!{sys.executable} -m src.train --config configs/deberta.yaml --overrides configs/smoke.yaml --fresh

What the trial says about the full run: memory, and a time estimate from its speed.

In [ ]:
import json, math
from src.config import load_config

full = load_config("configs/deberta.yaml")
trial = json.load(open("outputs/smoke/train_summary.json"))

seconds_per_step = trial["runtime_seconds"] / trial["steps"]
windows_full = trial["train_windows"] / 0.01                    # the trial used 1% of the data
per_update = full.training.batch_size * full.training.gradient_accumulation_steps
steps_full = math.ceil(windows_full / per_update) * full.training.num_epochs
hours = steps_full * seconds_per_step / 3600

print(f"Trial: {trial['steps']} steps in {trial['runtime_seconds']:.0f} s ({seconds_per_step:.2f} s per step), "
      f"peak GPU memory {trial.get('peak_gpu_memory_gb')} GB of about 15")
print(f"Losses: {[round(loss, 2) for _, loss in trial['losses']]}")
print(f"Full run: about {steps_full:,} steps, so roughly {hours:.1f} hours (the trial includes start-up time, "
      "so this is on the high side)")
print("RESULT:", "PASS" if trial["passed"] else "FAIL")

---
# Full run

## 2. Before you start

1. **The trial passed**, with no `nan` losses.
2. **Hugging Face token:** the **key icon** in Colab's left sidebar must have a secret named `HF_TOKEN` (Write
   permission) with "Notebook access" on. Same as for DistilBERT. Never paste the token into a cell.
3. **Google Drive:** about **5 GB free**. One checkpoint is about 2 GB, and replaced ones go to the Drive
   trash, which still counts: empty it if Drive fills up. The old DistilBERT checkpoints in
   `qa-project/distilbert-squad2` can be deleted (that model is on the Hub).
4. **Keep the tab open.** Free Colab stops idle sessions, and also stops long sessions after some hours.

**If Colab disconnects:** reconnect, run section 0 again, then sections 2 and 3. Training continues from the
newest checkpoint on Drive (saved every 1,000 steps) instead of starting over.

In [ ]:
import shutil

RUN_DIR = "/content/drive/MyDrive/qa-experiments/deberta-v3-squad2"   # checkpoints and the final model

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # src/train.py reads the token from here
else:
    RUN_DIR = "outputs/deberta-v3-squad2"

os.makedirs(RUN_DIR, exist_ok=True)
print(f"Saving to: {RUN_DIR}")
print(f"Free space there: {shutil.disk_usage(RUN_DIR).free / 1e9:.1f} GB (the run needs about 5)")
print("Hugging Face token found:", bool(os.environ.get("HF_TOKEN")))
print("Checkpoints already there:", sorted(p for p in os.listdir(RUN_DIR) if p.startswith("checkpoint-")) or "none (a new run)")

## 3. Train

All of `configs/deberta.yaml`: every training question, 2 epochs. It first checks that it can upload to the
Hub, so a bad token fails in seconds. The loss is printed every 100 steps; it ends with **RESULT: PASS** and
`Uploaded to https://huggingface.co/Ingsyling/deberta-v3-squad2`.

No `--fresh` on purpose: running this cell again continues an interrupted run.

In [ ]:
!{sys.executable} -m src.train --config configs/deberta.yaml --output-dir "{RUN_DIR}" --push-to-hub

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

run = json.load(open(f"{RUN_DIR}/train_summary.json"))
curve = pd.DataFrame(run["losses"], columns=["step", "loss"]).set_index("step")
ax = curve["loss"].plot(color="#2a78d6", alpha=0.35, figsize=(8, 3.5), label="every 100 steps")
curve["loss"].rolling(10, min_periods=1).mean().plot(ax=ax, color="#2a78d6", label="average of the last 10")
ax.set_title(f"DeBERTa-v3 training loss ({run['steps']:,} steps)")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False); plt.tight_layout(); plt.savefig(f"{RUN_DIR}/loss_curve.png", dpi=150); plt.show()
print(f"Steps: {run['steps']:,} | last loss: {curve['loss'].iloc[-1]:.3f} | RESULT:", "PASS" if run["passed"] else "FAIL")

## 4. Score it on the dev set

All 11,873 dev questions with the same script as before (a few minutes on a T4). To beat: RoBERTa's
**79.90 EM / 82.91 F1**.

In [ ]:
!{sys.executable} -m src.evaluate --model "{RUN_DIR}" --name deberta-v3-squad2 --output-dir "{RUN_DIR}/results"

## Next

Send the output of the trial (section 1), and after the full run the end of section 3 and the score table from
section 4. The scores and the predictions file (`results/deberta-v3-squad2_predictions.json` on Drive) then go
into the repo, and the threshold sweep and error analysis can be rerun for the new model.